# Day 26 — Dashboard Route and API Integration

This notebook inspects the existing Django URL configuration and adds a separate dashboard view plus a URL patch. It does not overwrite existing `views.py` or `urls.py`. Run from the repository root in the `ipl-player-performance` environment.

Day 25 created `dashboard_day25.html`, `dashboard_day25.css`, and `dashboard_day25.js`. The route and API must still be verified in a browser.

In [1]:
from pathlib import Path
import subprocess, sys, shutil

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    roots = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if roots: REPO_ROOT = roots[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
URLS_PY = APP_DIR / "urls.py"
TEMPLATE = APP_DIR / "templates" / "ipl_api" / "dashboard_day25.html"
assert MANAGE_PY.exists(), f"Missing {MANAGE_PY}"
assert URLS_PY.exists(), f"Missing {URLS_PY}"
assert TEMPLATE.exists(), f"Run Day 25 first; missing {TEMPLATE}"
print("Repository:", REPO_ROOT)
print("Existing urls.py:", URLS_PY)
print("Dashboard template:", TEMPLATE)
print("\n--- Existing urls.py (read-only preview) ---")
print(URLS_PY.read_text(encoding="utf-8"))


Repository: D:\Git\Personal\ipl-player-performance
Existing urls.py: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
Dashboard template: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html

--- Existing urls.py (read-only preview) ---
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView

urlpatterns = [
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),
]



In [2]:
view_source = 'from django.shortcuts import render\n\ndef dashboard_day25(request):\n    """Render the IPL analytics dashboard shell."""\n    return render(request, "ipl_api/dashboard_day25.html")\n'
compile(view_source, "dashboard_views_day26.py", "exec")
view_path = APP_DIR / "dashboard_views_day26.py"
if view_path.exists():
    shutil.copy2(view_path, view_path.with_suffix(".py.day26_backup"))
view_path.write_text(view_source, encoding="utf-8")
print("Created:", view_path)


Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\dashboard_views_day26.py


## Final root URL repair and automatic documentation
This cell deterministically replaces any existing root `path('', ...)` declaration, registers the root redirect, runs Django checks, resolves the actual URLs using Django's runtime resolver, runs the Day 21 regression tests, and generates `docs/day26_api_dashboard_repair_report.md`.


In [3]:
from pathlib import Path
import re, shutil, datetime, subprocess, sys

HERE = Path.cwd().resolve()
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "django_api" / "manage.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the IPL project repository.")
DJANGO_ROOT = REPO_ROOT / "django_api"
PROJECT_URLS = DJANGO_ROOT / "config" / "urls.py"
APP_URLS = DJANGO_ROOT / "ipl_api" / "urls.py"
MANAGE_PY = DJANGO_ROOT / "manage.py"
APP_DIR = DJANGO_ROOT / "ipl_api"
DOCS = REPO_ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)
REPORT = DOCS / "day26_api_dashboard_repair_report.md"
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

if not PROJECT_URLS.is_file() or not APP_URLS.is_file():
    raise FileNotFoundError(f"Expected URL files not found: {PROJECT_URLS} and/or {APP_URLS}")

def backup(path):
    dest = path.with_name(path.name + ".day26_verified_backup_" + stamp)
    shutil.copy2(path, dest)
    print("Backup created:", dest)

# Preserve project URL routes while ensuring the root redirect points to the dashboard.
project_before = PROJECT_URLS.read_text(encoding="utf-8")
project_text = project_before
if not re.search(r"(?m)^\s*from\s+django\.urls\s+import\s+.*\bpath\b", project_text):
    raise RuntimeError("config/urls.py does not import path from django.urls. No route was changed.")
if "RedirectView" not in project_text:
    project_text = re.sub(
        r"(?m)^(from django\.urls import .+)$",
        r"\1\nfrom django.views.generic.base import RedirectView",
        project_text, count=1
    )
if not re.search(r"(?m)^\s*urlpatterns\s*=\s*\[", project_text):
    raise RuntimeError("Could not find urlpatterns = [ in config/urls.py. No route was changed.")
# Remove duplicate root path declarations only; keep non-root project routes.
lines = project_text.splitlines()
clean_lines = []
root_seen = False
for line in lines:
    if re.match(r"^\s*path\(\s*(['\"])\s*\1\s*,", line):
        if root_seen:
            continue
        root_seen = True
        clean_lines.append('    path("", RedirectView.as_view(url="/api/dashboard/", permanent=False), name="home-dashboard"),')
    else:
        clean_lines.append(line)
project_text = "\n".join(clean_lines) + ("\n" if project_before.endswith("\n") else "")
if not root_seen:
    m = re.search(r"(?m)^\s*urlpatterns\s*=\s*\[", project_text)
    project_text = project_text[:m.end()] + '\n    path("", RedirectView.as_view(url="/api/dashboard/", permanent=False), name="home-dashboard"),' + project_text[m.end():]
compile(project_text, str(PROJECT_URLS), "exec")

# Merge required app routes without replacing unrelated routes. Only import symbols from files that exist.
app_before = APP_URLS.read_text(encoding="utf-8")
app_text = app_before
route_specs = [
    ("dashboard_views_day26.py", "dashboard_day25", "from .dashboard_views_day26 import dashboard_day25", 'path("dashboard/", dashboard_day25, name="ipl-dashboard")'),
    ("eda_summary_day26.py", "eda_summary_day26", "from .eda_summary_day26 import eda_summary_day26", 'path("eda/summary/", eda_summary_day26, name="eda-summary-day26")'),
]
for filename, symbol, import_line, path_line in route_specs:
    module_path = APP_DIR / filename
    if not module_path.is_file():
        print("Route not added; implementation file missing:", module_path)
        continue
    module_source = module_path.read_text(encoding="utf-8")
    if not re.search(r"\b" + re.escape(symbol) + r"\b", module_source):
        print("Route not added; view symbol not found:", symbol, "in", module_path)
        continue
    # Add import only if symbol isn't already imported.
    if symbol not in app_text:
        import_match = re.search(r"(?m)^from django\.urls import .+$", app_text)
        if import_match:
            app_text = app_text[:import_match.end()] + "\n" + import_line + app_text[import_match.end():]
        else:
            app_text = import_line + "\n" + app_text
    # Add route only if exact route fragment isn't already registered.
    route_fragment = path_line.split('"')[1]
    if not re.search(r"path\(\s*['\"]" + re.escape(route_fragment) + r"['\"]", app_text):
        urlpatterns_match = re.search(r"(?m)^\s*urlpatterns\s*=\s*\[", app_text)
        if not urlpatterns_match:
            raise RuntimeError("Could not find urlpatterns = [ in ipl_api/urls.py. No app URL write performed.")
        app_text = app_text[:urlpatterns_match.end()] + "\n    " + path_line + "," + app_text[urlpatterns_match.end():]

compile(app_text, str(APP_URLS), "exec")
if project_text != project_before:
    backup(PROJECT_URLS)
    PROJECT_URLS.write_text(project_text, encoding="utf-8")
    print("Updated project URL file:", PROJECT_URLS)
else:
    print("Project URL file unchanged.")
if app_text != app_before:
    backup(APP_URLS)
    APP_URLS.write_text(app_text, encoding="utf-8")
    print("Merged dashboard/EDA routes into:", APP_URLS)
else:
    print("App URL file already contains the required routes.")

project_text = PROJECT_URLS.read_text(encoding="utf-8")
app_text = APP_URLS.read_text(encoding="utf-8")
root_ok = bool(re.search(r'path\(\s*["\']\s*["\']\s*,\s*RedirectView\.as_view\(\s*url=["\']/api/dashboard/["\']', project_text))
dashboard_ok = bool(re.search(r'path\(\s*["\']dashboard/["\']', app_text)) and "dashboard_day25" in app_text
summary_ok = bool(re.search(r'path\(\s*["\']eda/summary/["\']', app_text)) and "eda_summary_day26" in app_text
print("Root redirect source check:", "PASS" if root_ok else "FAIL")
print("Dashboard app route source check:", "PASS" if dashboard_ok else "FAIL")
print("EDA summary app route source check:", "PASS" if summary_ok else "FAIL")

check = subprocess.run([sys.executable, str(MANAGE_PY), "check"], cwd=str(DJANGO_ROOT), capture_output=True, text=True)
print("\n$ python manage.py check\nExit code:", check.returncode)
print(check.stdout)
if check.stderr: print(check.stderr)

resolver_code = (
    "import os; os.environ.setdefault('DJANGO_SETTINGS_MODULE','config.settings'); "
    "import django; django.setup(); "
    "from django.urls import resolve; "
    "print('ROOT_RESOLVES_TO=' + str(resolve('/').url_name)); "
    "print('DASHBOARD_RESOLVES_TO=' + str(resolve('/api/dashboard/').url_name)); "
    "print('SUMMARY_RESOLVES_TO=' + str(resolve('/api/eda/summary/').url_name))"
)
resolve_result = subprocess.run([sys.executable, "-c", resolver_code], cwd=str(DJANGO_ROOT), capture_output=True, text=True)
print("\n$ Django URL resolver verification\nExit code:", resolve_result.returncode)
print(resolve_result.stdout)
if resolve_result.stderr: print(resolve_result.stderr)

tests = subprocess.run(
    [sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "1"],
    cwd=str(DJANGO_ROOT), capture_output=True, text=True
)
print("\n$ Day 21 regression tests\nExit code:", tests.returncode)
print(tests.stdout)
if tests.stderr: print(tests.stderr)

runtime_ok = (
    resolve_result.returncode == 0
    and "ROOT_RESOLVES_TO=home-dashboard" in resolve_result.stdout
    and "DASHBOARD_RESOLVES_TO=ipl-dashboard" in resolve_result.stdout
    and "SUMMARY_RESOLVES_TO=eda-summary-day26" in resolve_result.stdout
)
report = f"""# Day 26 — Dashboard and API URL Repair

Generated: {datetime.datetime.now().isoformat(timespec="seconds")}

## Verified results
- Root route source check: {"PASS" if root_ok else "FAIL"}
- Dashboard route source check: {"PASS" if dashboard_ok else "FAIL"}
- EDA summary route source check: {"PASS" if summary_ok else "FAIL"}
- Django URL resolver runtime check: {"PASS" if runtime_ok else "FAIL"}
- `python manage.py check`: {"PASS" if check.returncode == 0 else "FAIL"} (exit code {check.returncode})
- Day 21 regression tests: {"PASS" if tests.returncode == 0 else "FAIL"} (exit code {tests.returncode})

## Routes
- `/` should redirect to `/api/dashboard/`
- `/api/dashboard/` should resolve to `ipl-dashboard`
- `/api/eda/summary/` should resolve to `eda-summary-day26`

## Limitations
Passing route checks does not prove template rendering, MySQL queries, or browser JavaScript. Test the page in the browser after restarting the server.

## Files inspected
- `django_api/config/urls.py`
- `django_api/ipl_api/urls.py`
- `django_api/ipl_api/dashboard_views_day26.py`
- `django_api/ipl_api/eda_summary_day26.py`
"""
REPORT.write_text(report, encoding="utf-8")
print("\nDocumentation generated:", REPORT)
if not (root_ok and dashboard_ok and summary_ok and check.returncode == 0 and runtime_ok and tests.returncode == 0):
    raise RuntimeError("One or more checks failed. Read outputs and the generated report.")


Project URL file unchanged.
Backup created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py.day26_verified_backup_20261023_214317
Merged dashboard/EDA routes into: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
Root redirect source check: PASS
Dashboard app route source check: PASS
EDA summary app route source check: PASS

$ python manage.py check
Exit code: 0
System check identified no issues (0 silenced).


$ Django URL resolver verification
Exit code: 0
ROOT_RESOLVES_TO=home-dashboard
DASHBOARD_RESOLVES_TO=ipl-dashboard
SUMMARY_RESOLVES_TO=eda-summary-day26


$ Day 21 regression tests
Exit code: 0
Found 6 test(s).
System check identified no issues (0 silenced).

......
----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK


Documentation generated: D:\Git\Personal\ipl-player-performance\docs\day26_api_dashboard_repair_report.md


In [4]:
# Generate a compact per-notebook summary after the repair/validation cell has run.
report = f"""# Day 26 — Dashboard Route Integration

## Generated files
- `django_api/ipl_api/dashboard_views_day26.py`
- `django_api/config/urls.py` (updated only if needed)
- `django_api/ipl_api/urls.py` (dashboard and EDA summary routes merged only if needed)

## Automated checks
- `manage.py check` return code: {check.returncode}.
- URL resolver return code: {resolve_result.returncode}.
- Day 21 regression test return code: {tests.returncode}.

## Manual integration still required
1. Run `python manage.py runserver` from `django_api`.
2. Open `http://127.0.0.1:8000/api/dashboard/`.
3. Confirm the Day 26 dashboard shell appears.
4. Test the summary request at `http://127.0.0.1:8000/api/eda/summary/`.
5. Review the generated detailed report at `docs/day26_api_dashboard_repair_report.md`.

Automated checks do not prove database queries, model inference, or browser interactions succeed.
"""
doc_path = REPO_ROOT / "docs" / "day26_dashboard_api_integration.md"
doc_path.write_text(report, encoding="utf-8")
print("Saved:", doc_path)


Saved: D:\Git\Personal\ipl-player-performance\docs\day26_dashboard_api_integration.md


## Manual test checklist

- [ ] Run all cells in order from the repository root.
- [ ] Confirm `manage.py check`, URL resolution, and regression tests pass.
- [ ] Open `/api/dashboard/` and confirm it renders the Day 26 dashboard template.
- [ ] Open `/api/eda/summary/` and confirm it returns JSON.
- [ ] Preserve `/api/predictions/` and other existing API routes.
- [ ] Inspect `docs/day26_api_dashboard_repair_report.md` for actual recorded results.

Do not mark the task complete if any automated check fails. Browser and database behavior must still be tested manually.
